# How to Align an Engraved Score with a Recording and a Drawn Time Axis

Three files often travel together when a score is studied against a
performance:

- the **engraved score**, here an MEI file, read through
  [Verovio](https://www.verovio.org) by `VerovioLoader`;
- a **note alignment** between that score and a recording, in the MAPS
  format that the TROMPA project's
  [trompa-align](https://github.com/trompamusic/trompa-align) writes, read
  by `MapsLoader` as {{< glossary MatchClaim >}}s;
- a **drawing** of the score shifted to the recording's time, with a time
  axis below it, as [ScoreWarp](https://github.com/iwk-digital/scorewarp)
  writes it into an SVG, read by `ScoreWarpLoader`.

Each file states its time in its own unit: quarters for the score, seconds
for the recording, CSS pixels for the drawing. This guide loads all three
and answers one question across them: which score position lies under a
point of the drawing?

The score comes from the library's test corpus. The alignment and the
drawing are **illustrative**: this guide writes both itself, a few rows and
a few ticks, with invented times. They have the shape the two tools write,
so the same calls read the real files.

The arc:

1. Load the score: exact quarters, notes by `xml:id`, Verovio's own
   seconds.
2. Write a MAPS file, state it as MatchClaims, and ask a bundle for a
   {{< glossary MatchStamp >}} between two aligned notes.
3. Write a ScoreWarp SVG, read its time axis, and take a point of the
   drawing to seconds and on to the score.
4. Calibrate a drawn axis by hand with `create_calibrated_child`.
5. Save the bundle to a file and read it back.

## Setup

`VerovioLoader` needs the `verovio` extra: install the library with
`pip install "timetoalign[verovio]"`. The files this guide writes go to a
temporary directory. Three helpers keep the cells short: `notes_at` lists
the score's notes in a span of quarters, `ids_at` names the notes starting
at one position, and `claims_table` shows a list of claims one row each.

In [1]:
from __future__ import annotations

import json
import tempfile
from fractions import Fraction
from pathlib import Path

import pandas as pd

from timetoalign import AlignmentBundle, MapsLoader, ScoreWarpLoader, VerovioLoader
from timetoalign.testdata import ensure_data
from timetoalign.timelines import (
    ContinuousGraphicalTimeline,
    ContinuousPhysicalTimeline,
)

SCORE_PATH = (
    ensure_data("score")
    / "flow_control"
    / "out_of_the_flow_experience-polyrhythm_only.mei"
)
FILES = Path(tempfile.mkdtemp())

NOTE_COLUMNS = ["id", "start", "end", "mc", "staff", "midi"]


def notes_at(timeline, start, stop) -> pd.DataFrame:
    """The notes of *timeline* starting in ``[start, stop)`` quarters."""
    notes = timeline.get_events(include_children=True, event_type="Note").to_dataframe()
    span = notes[(notes["start"] >= start) & (notes["start"] < stop)]
    return span[NOTE_COLUMNS].reset_index(drop=True)


def ids_at(timeline, position) -> list[str]:
    """The ``xml:id`` of every note of *timeline* starting at *position*."""
    return list(notes_at(timeline, position, position + Fraction(1, 1000))["id"])


def claims_table(claims) -> pd.DataFrame:
    """One row per claim: its type, the event it names, and both sides."""
    rows = []
    for claim in claims:
        if claim.is_synchronous:
            here, there = (
                claim.start_anchor.coordinate_a,
                claim.start_anchor.coordinate_b,
            )
        else:
            here, there = claim.source_coordinate, None
        rows.append(
            {
                "claim_type": claim.claim_type.value,
                "event": claim.event_a_id,
                "from": claim.timeline_a_id,
                "at": str(here),
                "to": claim.timeline_b_id,
                "there": "" if there is None else str(there),
            }
        )
    return pd.DataFrame(rows)

## 1. Load the score

`VerovioLoader` reads any format Verovio reads (MEI, MusicXML, Humdrum, …)
and takes the score's time from Verovio's *timemap*, which states every
onset as an exact fraction of a quarter. The test score sets tuplets
against plain sixteenths: seven eighths in the time of three, each 3/14 of
a quarter long, and five sixteenths in the time of three, each 3/20. Its
positions are therefore multiples of 3/14 and of 3/20 of a quarter, such as
3/20, 3/14 and 3/10 in the first quarter: exact fractions keep them apart.

`create_timeline()` builds the logical timeline in quarters. Every movement
(`<mdiv>`) of the file is a child timeline holding that movement's events;
this score has one.

In [2]:
loader = VerovioLoader.from_file(SCORE_PATH)
score = loader.create_timeline(name="Out-of-the-flow Experience")
score

ContinuousLogicalTimeline(id='score:clt_1', length=105/2, unit=quarters, events=0, children=1, cmaps=1)

Every note keeps the `xml:id` it has in the MEI file; these ids were
written by the notation program that exported the score. The first
quarter, all four staves:

In [3]:
notes_at(score, 0, 1)

,id,start,end,mc,staff,midi
0,mscore-k1Qst3thRkC_3CANSxTdD9G,0,3/14,1,1,83
1,mscore-b5f5Huz.ns_lqEWpcW5F.E,0,1/4,1,2,51
2,mscore-MV3fqk.vEe_8h7vybZ51LJ,0,1/4,1,3,47
3,mscore-jaeveMoGy1F_QIFFzKi7IbP,0,3/20,1,4,32
4,mscore-I29VvVnnC9D_h457UM-XXuB,3/20,3/10,1,4,39
5,mscore-ux1ktDgRbPN_HhRlJ53e.PO,3/14,3/7,1,1,80
6,mscore-OKChrGsO4PG_jmEA-4XouzF,1/4,1/2,1,2,56
7,mscore-3OhQaTuj5yM_HWErHy.sfZO,1/4,1/2,1,3,51
8,mscore-Ab1ldf-xqoK_RFn.26ptQpF,3/10,9/20,1,4,44
9,mscore-TUDkCzWVjwK_835JqkZwz3D,3/7,9/14,1,1,75


An id is a key: `get_coordinate_for` gives a note's position, and
`get_timestamp_for` its span on the score and on its movement. The
timestamp of a position also shows the score's
{{< glossary ConversionMap display="conversion map" >}} to seconds, which
the loader builds from Verovio's own timing: the time Verovio renders from
the score's tempo, not a performance.

In [4]:
first_beat = ids_at(score, 1)[0]
score.get_coordinate_for(first_beat)

IdCoordinate(Fraction(1, 1), quarters, 'score:clt_1')

In [5]:
print(score.get_timestamp_for(first_beat))

TimeIntervalStamp [1, 5/4) quarters
  score:clt_1  1  5/4 quarters
  score:clt_2  1  5/4 quarters


In [6]:
score.get_timestamp_at(1)

ID,Coordinate,Type
score:clt_1,1 quarters,axis
score:clt_2,1 quarters,child
seconds,8/5 seconds,cmap


The same rendered time is available as a timeline of its own, in seconds,
with the same events and ids:

In [7]:
rendered = loader.create_physical_timeline()
rendered.get_coordinate_for(first_beat)

IdCoordinate(Fraction(8, 5), seconds, 'rendered:cpt_1')

## 2. A recording, aligned note by note

A MAPS file is a JSON list of rows. Each row pairs the score notes heard
together, named by `xml:id`, with one time of the recording in seconds,
`obs_mean_onset`; `obs_num` numbers the rows from 1. trompa-align writes two
more kinds of rows: a score note that was not played, at onset `-1`, and a
played note the score does not have, under an id beginning
`trompa-align_inserted_`.

The file below aligns the notes on the sixteenth-note grid of the first
measures, one unplayed note and one inserted note. Its times are invented.

In [8]:
onsets = {
    Fraction(0): 0.5,
    Fraction(1, 4): 0.88,
    Fraction(1, 2): 1.27,
    Fraction(3, 4): 1.69,
    Fraction(1): 2.08,
    Fraction(3, 2): 2.9,
}
rows = [{"obs_mean_onset": -1, "xml_id": ids_at(score, Fraction(3, 20))}]
rows += [
    {"obs_mean_onset": seconds, "xml_id": ids_at(score, quarter)}
    for quarter, seconds in onsets.items()
]
rows.insert(5, {"obs_mean_onset": 1.85, "xml_id": "trompa-align_inserted_Cs5"})
for number, row in enumerate(rows, start=1):
    row["obs_num"] = number

maps_path = FILES / "alignment.json"
maps_path.write_text(json.dumps(rows, indent=1), encoding="utf-8")
maps = MapsLoader.from_file(maps_path)
maps

Sources,1 file(s): alignment.json
Rows,8
Try,"create_bundle(score, performance), create_match_claims(score, performance)"


`create_match_claims(score, performance)` states every listed id as a
claim of its own. A note at a time of the recording becomes a synchronous
claim (`projection`: the score side names an event, the recording side a
time); the unplayed note and the inserted note each become a
{{< glossary NOMATCH >}} claim, from the side that has the note.

The recording is a timeline in seconds; any physical timeline works, and
one in samples reaches seconds through its C-Map.

In [9]:
recording = ContinuousPhysicalTimeline(length=4.0, unit="seconds", name="Recording")
claims = maps.create_match_claims(score, recording)
claims_table(claims)

,claim_type,event,from,at,to,there
0,nomatch,mscore-I29VvVnnC9D_h457UM-XXuB,score:clt_1,3/20 quarters,cpt_1,
1,projection,mscore-k1Qst3thRkC_3CANSxTdD9G,score:clt_1,0 quarters,cpt_1,0.5 seconds
2,projection,mscore-b5f5Huz.ns_lqEWpcW5F.E,score:clt_1,0 quarters,cpt_1,0.5 seconds
3,projection,mscore-MV3fqk.vEe_8h7vybZ51LJ,score:clt_1,0 quarters,cpt_1,0.5 seconds
4,projection,mscore-jaeveMoGy1F_QIFFzKi7IbP,score:clt_1,0 quarters,cpt_1,0.5 seconds
5,projection,mscore-OKChrGsO4PG_jmEA-4XouzF,score:clt_1,1/4 quarters,cpt_1,0.88 seconds
6,projection,mscore-3OhQaTuj5yM_HWErHy.sfZO,score:clt_1,1/4 quarters,cpt_1,0.88 seconds
7,projection,mscore-z3DiAbBuOnN_OuqIv8xBuID,score:clt_1,1/2 quarters,cpt_1,1.27 seconds
8,projection,mscore-VSZxPLX4fy_lz-c.-tBLBM,score:clt_1,1/2 quarters,cpt_1,1.27 seconds
9,projection,mscore-qaTSGF8H1nF_LsTiWUkHyVC,score:clt_1,3/4 quarters,cpt_1,1.69 seconds


A bundle relates the two timelines through the claims. Added with
`add_timeline()` alone, each timeline stands in no group, and the bundle
still builds its {{< glossary WarpMap >}} between them. A matchstamp at an
aligned note gives the recording's time; one at a note between two aligned
notes, such as the septuplet note at 3/7 of a quarter, gives a time
interpolated by the warp:

In [10]:
bundle = AlignmentBundle(name="Out-of-the-flow Experience")
bundle.add_timeline(score)
bundle.add_timeline(recording)
bundle.add_match_claims(claims)
bundle

AlignmentBundle(id='bundle_1', name='Out-of-the-flow Experience', timelines=2, groups=0)

In [11]:
bundle.get_matchstamp_at(Fraction(1, 4), score.id), bundle.get_matchstamp_at(
    Fraction(3, 7), score.id
)

(MatchStamp(score:clt_1=1/4 quarters, cpt_1=0.88 seconds),
 MatchStamp(score:clt_1=3/7 quarters, cpt_1=1.1585714285714286 seconds))

`create_bundle(score, performance)` is the one-call form: it puts each
timeline in a group of its own, named `score` and `performance`, and gives
the same answers.

In [12]:
maps.create_bundle(score, recording).get_matchstamp_at(Fraction(3, 7), score.id)

ID,Coordinate,Type
score:clt_1,3/7 quarters,inferred
cpt_1,1.1585714285714286 seconds,inferred


## 3. A drawing with a time axis

ScoreWarp engraves the score with Verovio, shifts every note to its time in
the recording, and appends a time axis to the SVG: a `<g class="timeAxis">`
holding a horizontal `line.horizontalAxis`, a vertical line per tick, a
number under every labelled tick and one title, such as `Time (s)`, naming
the unit. The helper below writes such an axis, from 0 to 4 seconds at 100
pixels per second, starting 50 pixels from the left edge.

In [13]:
def scorewarp_svg(path: Path, *, left: float, per_second: float, seconds: int) -> Path:
    """Write an SVG holding a ScoreWarp-shaped time axis, ticks every half second."""
    right = left + per_second * seconds
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{right + 20}" height="120" '
        f'viewBox="0 0 {right + 20} 120" overflow="visible">',
        '<g class="timeAxis">',
        f'<line class="horizontalAxis" x1="{left}" x2="{right}" y1="100" y2="100"/>',
        '<text x="2" y="96">Time (s)</text>',
    ]
    for half in range(2 * seconds + 1):
        x = left + per_second * half / 2
        parts.append(
            f'<line x1="{x}" x2="{x}" y1="100" y2="{106 if half % 2 == 0 else 103}"/>'
        )
    for second in range(seconds + 1):
        parts.append(f'<text x="{left + per_second * second}" y="116">{second}</text>')
    parts.append("</g></svg>")
    path.write_text("\n".join(parts), encoding="utf-8")
    return path


svg_path = scorewarp_svg(FILES / "warped.svg", left=50.0, per_second=100.0, seconds=4)
warped = ScoreWarpLoader.from_file(svg_path)
warped.axis_segment, warped.calibration

((Coordinate(50.0, css_pixels), Coordinate(450.0, css_pixels)),
 [(Coordinate(50.0, css_pixels), Coordinate(Fraction(0, 1), seconds)),
  (Coordinate(150.0, css_pixels), Coordinate(Fraction(1, 1), seconds)),
  (Coordinate(250.0, css_pixels), Coordinate(Fraction(2, 1), seconds)),
  (Coordinate(350.0, css_pixels), Coordinate(Fraction(3, 1), seconds)),
  (Coordinate(450.0, css_pixels), Coordinate(Fraction(4, 1), seconds))])

The drawing becomes a graphical timeline in `css_pixels`, the unit of an
SVG's outer viewport (1/96 inch, a continuous unit unlike the discrete
`pixels` of an image). Its time axis is a child over the drawn line, named
by the title, carrying the C-Map from `css_pixels` to the title's unit,
fitted through the labelled ticks:

In [14]:
drawing = warped.create_timeline()
axis = drawing.get_child(drawing.list_children()[0])
drawing, axis.get_conversion_map("seconds")

(ContinuousGraphicalTimeline(id='cgt_1', length=470.0, unit=css_pixels, events=0, children=1),
 LinearMap(scalar=0.01, offset=0.0, source_unit=css_pixels, target_unit=seconds))

A point of the drawing, say a notehead ScoreWarp drew at x = 258, is a
position on the drawing; its timestamp reaches the axis child, 50 pixels
further left, and the child's C-Map gives the time:

In [15]:
drawing.get_timestamp_at(258)

ID,Coordinate,Type
cgt_1,258 css_pixels,axis
cgt_2,208 css_pixels,child
seconds,2.08 seconds,cmap


That time is a position on the recording, and the bundle takes it to the
score: the notes there are the ones the drawing shows at x = 258.

In [16]:
seconds = drawing.get_timestamp_at(258).get_conversion_for("seconds")
stamp = bundle.get_matchstamp_at(seconds, recording.id)
stamp

ID,Coordinate,Type
cpt_1,2.08 seconds,anchor
score:clt_1,1 quarters,anchor


In [17]:
quarter = stamp.get_coordinate_for(score.id, format="fraction")
notes_at(score, quarter, quarter + Fraction(1, 1000))

,id,start,end,mc,staff,midi
0,mscore-5Ds8eINZfyN_1DnfAR.ihvE,1,5/4,1,2,68


## 4. Calibrating an axis by hand

`ScoreWarpLoader` reads the axis it finds in CSS pixels as drawn; an axis in
another drawing, or one placed through a transform, is calibrated by hand
with `Timeline.create_calibrated_child`. It takes the child's length and
offset and a list of `(position, value)` pairs, such as the labelled ticks
of an axis, and fits the conversion map through them.

Here the drawing is in `user_units`, the coordinate system of an SVG
`viewBox`, whose size in inches only the document knows. The axis runs from
120 to 1920 user units, labelled every second:

In [18]:
page = ContinuousGraphicalTimeline(length=2000, unit="user_units", name="Page")
ticks = [(120, 0), (720, 1), (1320, 2), (1920, 3)]
page_axis = page.create_calibrated_child(
    1800, 120, ticks, value_unit="seconds", name="Time (s)"
)
page.get_timestamp_at(594)

ID,Coordinate,Type
cgt_1,594 user_units,axis
cgt_2,474 user_units,child
seconds,0.79 seconds,cmap


The pairs are checked against the map's law, a straight line by default,
drawn through the first and the last pair. Below, the tick at 720 is
mislabelled 1.5 seconds: the line from `(120, 0)` to `(1920, 3)` reaches 1.5
only at 1020, so that pair is refused before the child is created, and the
page keeps only the axis made above:

In [19]:
try:
    page.create_calibrated_child(
        1800, 120, [(120, 0), (720, 1.5), (1320, 2), (1920, 3)], value_unit="seconds"
    )
except ValueError as error:
    print(error)
page.list_children()

Calibration pair 1 (720.0, 1.5) lies off the line through the first and last pair, which reaches 1.5 at 1020.0.


['cgt_2']

## 5. Saving the bundle

The drawing joins the bundle like the other two timelines. `to_file()`
writes the bundle as a {{< glossary "TimeToShare manifest" >}}: the
timelines, their children and conversion maps, the score's measures. A
manifest holds no MatchClaims, and events only when asked for with
`events=True`; the claims stay in the MAPS file, which names the score's
events, so the events are written too.

In [20]:
bundle.add_timeline(drawing)
bundle_path = bundle.to_file(FILES / "out_of_the_flow.yaml", events=True)
text = bundle_path.read_text(encoding="utf-8").splitlines()
start = text.index("timelines:")
print("\n".join(text[start:][:24]))

timelines:
  -
    children:
      -
        conversion_maps:
          -
            id: lmap_1
            scalar: 0.01
            source_unit: css_pixels
            target_unit: seconds
            type: LinearMap
        events: []
        id: cgt_2
        length: 400.0
        locked: true
        name: "Time (s)"
        offset: 50.0
        unit: css_pixels
    events:
      -
        duration: 400.0
        end: 450.0
        event_type: Segment
        id: cgt_2


`from_file()` reads the bundle back. The MAPS file states the claims again
on the timelines read, and the bundle answers as before: the drawing's
point at x = 258 is still 2.08 seconds, and the septuplet note still warps
to the same time.

In [21]:
read = AlignmentBundle.from_file(bundle_path)
read.add_match_claims(
    MapsLoader.from_file(maps_path).create_match_claims(
        read.get_timeline(score.id), read.get_timeline(recording.id)
    )
)
read.get_timeline(drawing.id).get_timestamp_at(258), read.get_matchstamp_at(
    Fraction(3, 7), score.id
)

(TimeStamp(axis=258 css_pixels @cgt_1, source='cgt_1', seconds=2.08),
 MatchStamp(score:clt_1=3/7 quarters, cpt_1=1.1585714285714286 seconds))

## Next

- Claims, claim types and WarpMaps in depth: the
  [Alignment Bundles and MatchClaims](../tutorials/tut06_alignment_bundles.ipynb)
  tutorial.
- Drawn and scanned time axes in pixels:
  [How to Work with Graphical Timelines](how01_graphical_timelines.ipynb).
- Manifests and files:
  [How to Save and Load Timelines, Groups and Bundles](how02_group_and_bundle_files.ipynb).